# SARSA：更谨慎的表格型强化学习
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：SARSA.py | 核心功能：On-policy TD 学习、与 Q-Learning 对比、安全策略

## 概述

SARSA（State-Action-Reward-State-Action）是 on-policy 的 TD 学习算法。名字来源于它的更新需要五元组 (S, A, R, S', A')。

与 Q-Learning 的核心区别：Q-Learning 用 max Q(s',a') 更新（假想中的最优动作），SARSA 用实际选择的 Q(s',a') 更新。这让 SARSA 更保守——它在训练中考虑了探索的风险。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from collections import defaultdict

## 数学原理

### 1. 马尔可夫决策过程（MDP）

强化学习的数学框架是 MDP，定义为五元组 $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$：

- $\mathcal{S}$：状态空间
- $\mathcal{A}$：动作空间
- $P(s'|s, a)$：状态转移概率
- $R(s, a)$：奖励函数
- $\gamma \in [0, 1)$：折扣因子

**代码对应**：`gymnasium` 环境封装了 MDP 的所有元素。

### 2. 值函数与 Bellman 方程

**状态值函数**：$V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t | s_0 = s\right]$

**动作值函数**：$Q^\pi(s, a) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t | s_0 = s, a_0 = a\right]$

**Bellman 方程**（最优）：

$$Q^*(s, a) = R(s, a) + \gamma\sum_{s'}P(s'|s, a)\max_{a'}Q^*(s', a')$$

### 3. Q-Learning 算法

**代码对应**：Q-Learning 是**离策略**（off-policy）时序差分学习。

更新规则：

$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma\max_{a'}Q(s', a') - Q(s, a)\right]$$

其中 $\alpha$ 为学习率，$r + \gamma\max_{a'}Q(s', a')$ 为 **TD 目标**。

关键：$\max_{a'}Q(s', a')$ 使用**贪婪策略**选择动作，而实际执行的动作由 $\epsilon$-贪婪策略决定（离策略）。

### 4. SARSA 算法

**代码对应**：SARSA 是**同策略**（on-policy）时序差分学习。

更新规则：

$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma Q(s', a') - Q(s, a)\right]$$

其中 $a'$ 是**实际执行**的动作（由当前策略选择）。

### 5. Q-Learning vs SARSA

| 特性 | Q-Learning | SARSA |
|------|-----------|-------|
| 策略类型 | 离策略 | 同策略 |
| TD 目标 | $r + \gamma\max_{a'}Q(s', a')$ | $r + \gamma Q(s', a')$ |
| 收敛到 | $Q^*$（最优 Q 函数） | $Q^\pi$（当前策略的 Q 函数） |
| 风险偏好 | 更激进（学习最优） | 更保守（考虑探索风险） |

**代码对应**：Q-Learning 更新中 `np.max(Q[next_state])` vs SARSA 中 `Q[next_state, next_action]`。

### 6. $\epsilon$-贪婪策略

$$\pi(a|s) = \begin{cases} 1 - \epsilon + \epsilon/|\mathcal{A}| & a = \arg\max_{a'}Q(s, a') \\ \epsilon/|\mathcal{A}| & \text{otherwise} \end{cases}$$

$\epsilon$ 控制探索与利用的平衡。通常从 $\epsilon = 1$ 逐步衰减到 $\epsilon_{\min}$。

### 1. 网格世界环境（同 Q_Learning.py）

运行 1. 网格世界环境（同 Q_Learning.py） 的代码，观察算法在该环节的行为。

In [ ]:
class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.goal = (3, 3)
        self.traps = {(1, 1), (2, 2)}
# --- 赋值/配置 ---
        self.actions = [0, 1, 2, 3]
        self.action_names = ["↑", "↓", "←", "→"]

    def reset(self):
        self.state = (0, 0)
        return self.state

    def step(self, action):
        r, c = self.state
        if action == 0: r = max(0, r - 1)
        elif action == 1: r = min(self.size - 1, r + 1)
        elif action == 2: c = max(0, c - 1)
# --- 条件判断 ---
        elif action == 3: c = min(self.size - 1, c + 1)
        self.state = (r, c)

        if self.state == self.goal:
            return self.state, 10.0, True
        elif self.state in self.traps:
            return self.state, -5.0, True
        return self.state, -0.1, False

    def render(self, q_table=None):
        symbols = np.full((self.size, self.size), ".", dtype=object)
        symbols[self.goal[0], self.goal[1]] = "G"
        for t in self.traps:
            symbols[t[0], t[1]] = "T"
# --- 条件判断 ---
        if q_table:
            for r in range(self.size):
                for c in range(self.size):
                    if (r, c) not in self.traps and (r, c) != self.goal:
                        q_vals = [q_table.get(((r, c), a), 0) for a in self.actions]
# --- 数值计算 ---
                        symbols[r, c] = self.action_names[np.argmax(q_vals)]
        print("\n".join([" ".join(row) for row in symbols]))

### 2. SARSA 算法

运行 2. SARSA 算法 的代码，观察算法在该环节的行为。

In [ ]:
def sarsa(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1, epsilon_decay=0.995):
    """SARSA 核心算法（on-policy）"""
    Q = defaultdict(float)
    episode_rewards = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False

        # 选择第一个动作
        if np.random.rand() < epsilon:
            action = np.random.choice(env.actions)
        else:
            action = np.argmax([Q[(state, a)] for a in env.actions])

        while not done:
            next_state, reward, done = env.step(action)
            total_reward += reward

            # 选择下一个动作（SARSA 的关键：用行为策略选下一步动作）
            if np.random.rand() < epsilon:
                next_action = np.random.choice(env.actions)
            else:
                next_action = np.argmax([Q[(next_state, a)] for a in env.actions])

            # SARSA 更新（on-policy：用实际选择的 A' 更新）
            Q[(state, action)] += alpha * (reward + gamma * Q[(next_state, next_action)] - Q[(state, action)])

            state, action = next_state, next_action

        episode_rewards.append(total_reward)
        epsilon *= epsilon_decay

    return Q, episode_rewards

### 3. Q-Learning 对比

对比不同模型或配置的性能差异。

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1, epsilon_decay=0.995):
    Q = defaultdict(float)
    episode_rewards = []
    for episode in range(n_episodes):
        state = env.reset()
# --- 赋值/配置 ---
        total_reward = 0
        done = False
        while not done:
            if np.random.rand() < epsilon:
                action = np.random.choice(env.actions)
# --- 继续 ---
            else:
                action = np.argmax([Q[(state, a)] for a in env.actions])
            next_state, reward, done = env.step(action)
            total_reward += reward
            best_next = max(Q[(next_state, a)] for a in env.actions)
# --- 继续 ---
            Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])
            state = next_state
        episode_rewards.append(total_reward)
        epsilon *= epsilon_decay
    return Q, episode_rewards

### 4. 训练与对比

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = GridWorld()
Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=1000, epsilon=0.2)
Q_ql, rewards_ql = q_learning(env, n_episodes=1000, epsilon=0.2)

print("=== SARSA vs Q-Learning ===")
print(f"SARSA  最后 100 平均奖励: {np.mean(rewards_sarsa[-100:]):.2f}")
print(f"Q-Learning 最后 100 平均奖励: {np.mean(rewards_ql[-100:]):.2f}")

### 5. 策略对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== SARSA 策略 ===")
env.render(Q_sarsa)
print("\n=== Q-Learning 策略 ===")
env.render(Q_ql)

### 6. 关键区别

运行 6. 关键区别 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== SARSA vs Q-Learning 核心区别 ===")
print("Q-Learning 更新: Q(s,a) += α × [r + γ × max Q(s',a') - Q(s,a)]")
print("SARSA 更新:      Q(s,a) += α × [r + γ × Q(s',a') - Q(s,a)]")
print("  其中 a' 是行为策略（ε-greedy）实际选择的动作，不是 greedy 的 max")
print()
# --- 输出结果 ---
print("效果差异:")
print("- Q-Learning 学到的是最优策略（冒险走捷径）")
print("- SARSA 学到的是更保守的策略（避开陷阱附近路径）")
print('- 因为 SARSA 在训练中"执行"了探索动作，更考虑探索风险')

print("\n=== SARSA 要点 ===")
print("- On-policy：行为策略 = 目标策略（都是 ε-greedy）")
print("- 比 Q-Learning 更保守（考虑探索时的风险）")
print("- 在有风险的环境中更安全（不走陷阱旁边）")
print("- 收敛速度可能比 Q-Learning 慢（因为受探索噪声影响）")

## 关键代码解释

### SARSA 更新

```python
# 先选下一个动作，再更新
next_action = epsilon_greedy(...)
Q[(state, action)] += alpha * (reward + gamma * Q[(next_state, next_action)] - Q[(state, action)])
```

SARSA 必须在更新前就选好下一步动作 A'，因为更新需要用到实际选的 A'。

### 策略差异

在有陷阱的环境中，Q-Learning 学到的策略会走陷阱旁边（因为最优路径更短），SARSA 学到的策略会远离陷阱（因为探索时可能掉进去）。

## 注意事项

1. **On-policy**：训练数据来自当前策略，策略改变后旧数据不可用
2. **更安全**：在有风险的环境中比 Q-Learning 更保守
3. **收敛更慢**：受探索噪声影响

## 总结与延伸

以上代码展示了 **SARSA** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Expected SARSA**：用 Q 值的期望替代实际采样的 A'，减少方差
- **SARSA(lambda)**：资格迹（Eligibility Trace），加速信用分配
- **安全 RL**：SARSA 的保守性启发了约束策略优化（CPO）
